# DATATHON 2026 — THE GRIDBREAKER
**Hosted by:** VinTelligence - VinUniversity Data Science & AI Club

---

## THÔNG TIN NHÓM
- **Tên đội thi**: Last Dance
- **Mã đội thi**: kVmzJpHWUFT6mv82wG4U
- **Thành viên**: 

| STT | Họ và tên        |  Vai trò   | Email |
| --- | ---------------- | ---------- | ----- |
| 1   | Bùi Lê Khôi      | Đội trưởng |       |
| 2   | Nguyễn Hà Anh    | Thành viên |       |
| 3   | Bùi Công Mậu     | Thành viên |       |
| 4   | Thái Hữu Thọ     | Thành viên |       |

---

## GIỚI THIỆU FILE NOTEBOOK

- **Mục đích**: Tạo features mới để chuẩn bị trong **Phần 2 — Trực quan hoá và Phân tích Dữ liệu**

- **Nội dung trong file**:
    - Kiểm tra tính hợp lệ của dữ liệu đầu vào.
    - Tạo các tập dữ liệu giàu tính năng: `enriched_customers`, `enriched_products`, `enriched_orders`.
    - Tính toán RFM (raw values + quintile scores), Return Rate an toàn, Shipping Duration.

- **Dữ liệu sử dụng**:
    - Dữ liệu đầu vào là file đầu ra sau khi chạy file `LD_02_DA_01_Preprocessing`.
    
---

Bài làm được trình bày từ sau dòng này.

---
---

### Import các libs và dependencies

In [8]:
import re
import os
import math

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import openpyxl as pxl
import seaborn as sns
import statsmodels.api as sm
import datetime as dt

from datetime import timedelta
from scipy import stats
from math import sqrt

from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, pairwise_distances, silhouette_score, precision_score, recall_score, f1_score
from sklearn.metrics import RocCurveDisplay, roc_auc_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

import warnings
warnings.filterwarnings("ignore")

# Ensure plots appear in the notebook
%matplotlib inline

# Print last updated timestamp
import time
print(f"Cập nhật lần cuối vào thời điểm {time.asctime()}")

Cập nhật lần cuối vào thời điểm Fri May  1 08:56:59 2026



### Load dữ liệu và kiểm tra tính hợp lệ

> **Mục đích**: Đảm bảo tất cả các file tồn tại, không rỗng, và có định dạng cột đúng trước khi chạy feature engineering.

In [9]:
DATA_PATH = '../data/processed/'

# --- Load tất cả các file ---
required_files = {
    'customers':   'customers.csv',
    'orders':      'orders.csv',
    'order_items': 'order_items.csv',
    'products':    'products.csv',
    'payments':    'payments.csv',
    'returns':     'returns.csv',
    'reviews':     'reviews.csv',
    'shipments':   'shipments.csv',
}

dfs = {}
for name, filename in required_files.items():
    path = os.path.join(DATA_PATH, filename)
    try:
        dfs[name] = pd.read_csv(path)
    except FileNotFoundError:
        raise FileNotFoundError(f'[MISSING] Không tìm thấy file: {path}')

customers   = dfs['customers']
orders      = dfs['orders']
order_items = dfs['order_items']
products    = dfs['products']
payments    = dfs['payments']
returns     = dfs['returns']
reviews     = dfs['reviews']
shipments   = dfs['shipments']



### Chuyển đổi kiểu dữ liệu ngày tháng

In [10]:
# Chuyển đổi các cột ngày tháng với errors='coerce' để tránh crash khi có giá trị lỗi
orders['order_date']          = pd.to_datetime(orders['order_date'],          errors='coerce')
customers['signup_date']      = pd.to_datetime(customers['signup_date'],      errors='coerce')
shipments['ship_date']        = pd.to_datetime(shipments['ship_date'],         errors='coerce')
shipments['delivery_date']    = pd.to_datetime(shipments['delivery_date'],     errors='coerce')

# Kiểm tra xem có ngày nào bị lỗi (NaT) không
date_checks = {
    'orders.order_date':           orders['order_date'].isnull().sum(),
    'customers.signup_date':       customers['signup_date'].isnull().sum(),
    'shipments.ship_date':         shipments['ship_date'].isnull().sum(),
    'shipments.delivery_date':     shipments['delivery_date'].isnull().sum(),
}
for col, n_null in date_checks.items():
    status = '⚠ CÓ GIÁ TRỊ LỖI' if n_null > 0 else '✓ OK'
    print(f'{col:<35}: NaT = {n_null:>4}  {status}')

snapshot_date = orders['order_date'].max() + pd.Timedelta(days=1)
print(f'\nSnapshot date: {snapshot_date.date()}')

orders.order_date                  : NaT =    0  ✓ OK
customers.signup_date              : NaT =    0  ✓ OK
shipments.ship_date                : NaT =    0  ✓ OK
shipments.delivery_date            : NaT =    0  ✓ OK

Snapshot date: 2023-01-01



###  Tạo `enriched_customers` — RFM + Tenure + RFM Scoring


In [11]:
# --- Bước 4.1: Tính RFM raw values ---
# FIX: thêm reset_index() để customer_id không bị kẹt là Index
customer_rfm = (
    orders.groupby('customer_id')
    .agg(
        recency   = ('order_date',  lambda x: (snapshot_date - x.max()).days),
        frequency = ('order_id',    'nunique')
    )
    .reset_index()   # <-- FIX: bắt buộc
)

# FIX: monetary cũng reset_index() trước khi merge
customer_monetary = (
    payments
    .merge(orders[['order_id', 'customer_id']], on='order_id', how='left')
    .groupby('customer_id')['payment_value']
    .sum()
    .rename('monetary')
    .reset_index()   # <-- FIX
)

# Merge từng bước — rõ ràng, dễ debug
customer_features = customer_rfm.merge(customer_monetary, on='customer_id', how='left')
customer_features['monetary'] = customer_features['monetary'].fillna(0)

# Tenure
customer_features = customer_features.merge(
    customers[['customer_id', 'signup_date']], on='customer_id', how='left'
)
customer_features['tenure_days'] = (
    snapshot_date - customer_features['signup_date']
).dt.days

# --- Bước 4.2: RFM Scoring (quintile 1–5) ---
# Recency: thấp = tốt => score cao => labels đảo ngược [5,4,3,2,1]
# Frequency & Monetary: cao = tốt => score cao => labels thuận [1,2,3,4,5]
def rfm_quintile(series, ascending=True):
    """Chia quintile, xử lý trùng lặp bằng rank(method='first')."""
    labels = [1, 2, 3, 4, 5] if ascending else [5, 4, 3, 2, 1]
    return pd.qcut(series.rank(method='first'), 5, labels=labels).astype(int)

customer_features['R_score']   = rfm_quintile(customer_features['recency'],   ascending=False)
customer_features['F_score']   = rfm_quintile(customer_features['frequency'],  ascending=True)
customer_features['M_score']   = rfm_quintile(customer_features['monetary'],   ascending=True)
customer_features['RFM_score'] = (
    customer_features['R_score'] +
    customer_features['F_score'] +
    customer_features['M_score']
)

# Lưu file
customer_features.to_csv(os.path.join(DATA_PATH, 'enriched_customers.csv'), index=False)

print(f'enriched_customers: {customer_features.shape[0]:,} rows, {customer_features.shape[1]} cols')
print(f'\nPhân phối RFM_score:')
print(customer_features['RFM_score'].describe().round(2))
customer_features[['recency', 'frequency', 'monetary', 'R_score', 'F_score', 'M_score', 'RFM_score']].head()

enriched_customers: 90,246 rows, 10 cols

Phân phối RFM_score:
count    90246.00
mean         9.00
std          3.81
min          3.00
25%          6.00
50%          9.00
75%         12.00
max         15.00
Name: RFM_score, dtype: float64


,recency,frequency,monetary,R_score,F_score,M_score,RFM_score
0,617,6,142803.47,4,4,4,12
1,179,4,204693.89,5,3,4,12
2,3443,3,52093.47,1,2,2,5
3,917,1,10939.06,3,1,1,5
4,1376,5,64179.86,3,3,3,9



### Tạo `enriched_products` — Return Rate + Average Rating


In [12]:
# --- Bước 5.1: Thống kê bán hàng và trả hàng ---
prod_sales   = order_items.groupby('product_id')['quantity'].sum().rename('total_sold').reset_index()
prod_returns = returns.groupby('product_id')['return_quantity'].sum().rename('total_returned').reset_index()

product_stats = prod_sales.merge(prod_returns, on='product_id', how='left')
product_stats['total_returned'] = product_stats['total_returned'].fillna(0)

# FIX: tránh division by zero
product_stats['return_rate'] = np.where(
    product_stats['total_sold'] > 0,
    product_stats['total_returned'] / product_stats['total_sold'],
    0.0
)

# --- Bước 5.2: Rating trung bình ---
prod_ratings = (
    reviews.groupby('product_id')['rating']
    .agg(avg_rating='mean', review_count='count')
    .reset_index()
)

# FIX: merge rõ ràng bằng on='product_id' thay vì right_index=True
enriched_products = (
    products
    .merge(product_stats, on='product_id', how='left')
    .merge(prod_ratings,  on='product_id', how='left')
)
enriched_products[['total_sold', 'total_returned', 'return_rate', 'avg_rating', 'review_count']] = (
    enriched_products[['total_sold', 'total_returned', 'return_rate', 'avg_rating', 'review_count']]
    .fillna(0)
)

enriched_products.to_csv(os.path.join(DATA_PATH, 'enriched_products.csv'), index=False)

print(f'enriched_products: {enriched_products.shape[0]:,} rows, {enriched_products.shape[1]} cols')
print(f'\nReturn rate summary:')
print(enriched_products['return_rate'].describe().round(4))
print(f'\nSản phẩm có return_rate > 10%: {(enriched_products["return_rate"] > 0.1).sum()} sản phẩm')
enriched_products.head()

enriched_products: 2,412 rows, 13 cols

Return rate summary:
count    2412.0000
mean        0.0233
std         0.0438
min         0.0000
25%         0.0000
50%         0.0122
75%         0.0348
max         0.8571
Name: return_rate, dtype: float64

Sản phẩm có return_rate > 10%: 57 sản phẩm


,product_id,product_name,category,segment,size,color,price,cogs,total_sold,total_returned,return_rate,avg_rating,review_count
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,Green,11059.650000,9704.842875,108.0,10.0,0.092593,4.000000,3.0
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,Silver,9523.076013,5393.870254,367.0,10.0,0.027248,3.687500,16.0
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,Pink,15951.633158,11371.919278,82.0,3.0,0.036585,3.666667,6.0
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,Yellow,15753.717299,8573.172954,676.0,22.0,0.032544,3.766667,30.0
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,Red,15766.334536,14063.570406,336.0,14.0,0.041667,3.555556,9.0


### Tạo `enriched_orders` — Shipping Duration + Order Features

In [13]:
# --- Bước 6.1: Shipping duration ---
shipments['shipping_duration'] = (
    shipments['delivery_date'] - shipments['ship_date']
).dt.days

# Kiểm tra giá trị âm (delivery trước ship — lỗi data)
negative_duration = (shipments['shipping_duration'] < 0).sum()
if negative_duration > 0:
    print(f'⚠ Có {negative_duration} đơn có shipping_duration âm — kiểm tra lại data!')
    shipments.loc[shipments['shipping_duration'] < 0, 'shipping_duration'] = np.nan

# --- Bước 6.2: Merge vào order master ---
order_master = orders.merge(
    shipments[['order_id', 'shipping_duration', 'shipping_fee']],
    on='order_id',
    how='left'
)

# Thêm các time features hữu ích
order_master['order_month']    = order_master['order_date'].dt.month
order_master['order_quarter']  = order_master['order_date'].dt.quarter
order_master['order_dayofweek']= order_master['order_date'].dt.dayofweek  # 0=Thứ Hai
order_master['order_year']     = order_master['order_date'].dt.year

order_master.to_csv(os.path.join(DATA_PATH, 'enriched_orders.csv'), index=False)

print(f'enriched_orders: {order_master.shape[0]:,} rows, {order_master.shape[1]} cols')
print(f'\nShipping duration summary (ngày):')
print(order_master['shipping_duration'].describe().round(2))
print(f'\nĐơn hàng giao > 5 ngày: {(order_master["shipping_duration"] > 5).sum():,}')

enriched_orders: 646,945 rows, 14 cols

Shipping duration summary (ngày):
count    566067.00
mean          4.50
std           1.71
min           2.00
25%           3.00
50%           4.00
75%           6.00
max           7.00
Name: shipping_duration, dtype: float64

Đơn hàng giao > 5 ngày: 188,519



### Kiểm tra tổng kết — Xác nhận các file enriched đã sẵn sàng

In [14]:
print('Kết quả Feature Engineering')
output_files = [
    ('enriched_customers.csv', customer_features),
    ('enriched_products.csv',  enriched_products),
    ('enriched_orders.csv',    order_master),
]
print(f'{"File":<28} {"Rows":>8} {"Cols":>6} {"NaN":>8}')
print('-' * 55)
for fname, df in output_files:
    print(f'{fname:<28} {df.shape[0]:>8,} {df.shape[1]:>6} {df.isnull().sum().sum():>8,}')


Kết quả Feature Engineering
File                             Rows   Cols      NaN
-------------------------------------------------------
enriched_customers.csv         90,246     10        0
enriched_products.csv           2,412     13        0
enriched_orders.csv           646,945     14  161,756
